In [1]:
from src.partition_tests import run_partition_tests
from src.metamorphic_tests import run_metamorphic_tests
from src.helpers import *

# Load Data and Models

In [2]:
DATA_PATH = "data/investigation_train_large_checked.csv"
ADDITIONAL_DATA_PATH = "data/synth_data_for_training.csv"

DUMMY_ONNX_PATH = "models/dummy_model.onnx"
GOOD_ONNX_PATH  = "models/good_model.onnx"
BAD_ONNX_PATH   = "models/bad_model.onnx"

In [3]:
df = pd.read_csv(DATA_PATH)
df_additional = pd.read_csv(ADDITIONAL_DATA_PATH)

leakage_cols = ["checked", "Ja", "Nee"]

y = df["checked"]
y_additional = df_additional["checked"]
X = df.drop(columns=leakage_cols, errors="ignore").fillna(0).astype(np.float32)
X_additional = df_additional.drop(columns=leakage_cols, errors="ignore").fillna(0).astype(np.float32)

print("X shape:", X.shape)
print("X_additional shape:", X_additional.shape)

X shape: (130000, 315)
X_additional shape: (12645, 315)


In [4]:
dummy_sess, dummy_in = load_onnx_session(DUMMY_ONNX_PATH)
good_sess, good_in   = load_onnx_session(GOOD_ONNX_PATH)
bad_sess, bad_in     = load_onnx_session(BAD_ONNX_PATH)

dummy_model = make_onnx_model(dummy_sess, dummy_in)
good_model  = make_onnx_model(good_sess, good_in)
bad_model   = make_onnx_model(bad_sess, bad_in)

# Performance Evaluation

## Dummy Model

In [5]:
evaluate_onnx(dummy_model, X, y, name="Dummy Logistic Regression (Original Data)")
evaluate_onnx(dummy_model, X, y, name="Dummy Logistic Regression (Additional Data)")


===== Evaluation: Dummy Logistic Regression (Original Data) =====
Accuracy : 0.8805153846153846
Precision: 0.9243428082923701
Recall   : 0.22174938474159148
F1-score : 0.35768928586196913
ROC AUC  : 0.9297284841431384

===== Evaluation: Dummy Logistic Regression (Additional Data) =====
Accuracy : 0.8805153846153846
Precision: 0.9243428082923701
Recall   : 0.22174938474159148
F1-score : 0.35768928586196913
ROC AUC  : 0.9297284841431384


## Good Model

In [6]:
evaluate_onnx(good_model, X, y, name="Good Model (Original Data)")
evaluate_onnx(good_model, X, y, name="Good Model (Additional Data)")


===== Evaluation: Good Model (Original Data) =====
Accuracy : 0.8575692307692308
Precision: 0.5909090909090909
Recall   : 0.16463289581624282
F1-score : 0.25751864624268184
ROC AUC  : 0.7755390371036635

===== Evaluation: Good Model (Additional Data) =====
Accuracy : 0.8575692307692308
Precision: 0.5909090909090909
Recall   : 0.16463289581624282
F1-score : 0.25751864624268184
ROC AUC  : 0.7755390371036635


## Bad Model

In [7]:
evaluate_onnx(bad_model, X, y, name="Bad Model (Original Data)")
evaluate_onnx(bad_model, X, y, name="Bad Model (Additional Data)")


===== Evaluation: Bad Model (Original Data) =====
Accuracy : 0.8592538461538461
Precision: 0.6190570132175972
Recall   : 0.16089007383100903
F1-score : 0.25540227078500793
ROC AUC  : 0.782967249309074

===== Evaluation: Bad Model (Additional Data) =====
Accuracy : 0.8592538461538461
Precision: 0.6190570132175972
Recall   : 0.16089007383100903
F1-score : 0.25540227078500793
ROC AUC  : 0.782967249309074


# Run Partition and Metamorphic Tests

## Dummy Original Data

In [8]:
run_partition_tests(dummy_model, X)


=== Language requirement met ===
Language requirement met: n=71680, mean risk=0.134
Language requirement not met: n=52185, mean risk=0.168
  [FAIL] Language requirement compliance has big impact on predicted risk.

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=34283, mean risk=0.152
Highest 20% language-related score: n=42352, mean risk=0.150
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=27316, mean risk=0.140
At least one attitude/motivation judgement recorded: n=102684, mean risk=0.153
  [OK] 

=== Subjective communication judgement ===
Has communication judgement recorded: n=68601, mean risk=0.157
No communication judgement recorded: n=61399, mean risk=0.143
  [OK] 

=== Subjective appearance/presentation judgements ===
No appearance/presentation judgement recorded: n=55300, mean risk=0.143
At least one appearance/presentation judgement recorded: n=74700, mean risk=0.156
  [OK] 

=== Subjec

In [9]:
run_metamorphic_tests(dummy_model, X)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.1506
Mean score (flipped)  :       0.1506
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.1506
Mean score (flipped)  :       0.1506
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.1506
Mean score (flipped)  :       0.1506
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Attitude/motivation: baseline -> NONE recorded ===
Mean score (baseline) :       0.1506
Mean score (flipped)  :       0.1506
Mean shift       

## Dummy Additional Data

In [10]:
run_partition_tests(dummy_model, X_additional)


=== Language requirement met ===
Language requirement met: n=7011, mean risk=0.132
Language requirement not met: n=5043, mean risk=0.169
  [FAIL] Language requirement compliance has big impact on predicted risk.

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=3350, mean risk=0.154
Highest 20% language-related score: n=4086, mean risk=0.149
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=2651, mean risk=0.141
At least one attitude/motivation judgement recorded: n=9994, mean risk=0.152
  [OK] 

=== Subjective communication judgement ===
Has communication judgement recorded: n=6643, mean risk=0.158
No communication judgement recorded: n=6002, mean risk=0.141
  [OK] 

=== Subjective appearance/presentation judgements ===
No appearance/presentation judgement recorded: n=5341, mean risk=0.142
At least one appearance/presentation judgement recorded: n=7304, mean risk=0.156
  [OK] 

=== Subjective behavi

In [11]:
run_metamorphic_tests(dummy_model, X_additional)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.1500
Mean score (flipped)  :       0.1500
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.1500
Mean score (flipped)  :       0.1500
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.1500
Mean score (flipped)  :       0.1500
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Attitude/motivation: baseline -> NONE recorded ===
Mean score (baseline) :       0.1500
Mean score (flipped)  :       0.1500
Mean shift       

## Good Original Data

In [12]:
run_partition_tests(good_model, X)


=== Language requirement met ===
Language requirement met: n=71680, mean risk=0.304
Language requirement not met: n=52185, mean risk=0.323
  [OK] 

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=34283, mean risk=0.323
Highest 20% language-related score: n=42352, mean risk=0.307
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=27316, mean risk=0.321
At least one attitude/motivation judgement recorded: n=102684, mean risk=0.312
  [OK] 

=== Subjective communication judgement ===
Has communication judgement recorded: n=68601, mean risk=0.311
No communication judgement recorded: n=61399, mean risk=0.316
  [OK] 

=== Subjective appearance/presentation judgements ===
No appearance/presentation judgement recorded: n=55300, mean risk=0.316
At least one appearance/presentation judgement recorded: n=74700, mean risk=0.312
  [OK] 

=== Subjective behavioural/capacity judments ===
No behavioural/capacity judg

In [13]:
run_metamorphic_tests(good_model, X)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.3136
Mean score (flipped)  :       0.3136
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.3136
Mean score (flipped)  :       0.3136
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.3136
Mean score (flipped)  :       0.3136
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Attitude/motivation: baseline -> NONE recorded ===
Mean score (baseline) :       0.3136
Mean score (flipped)  :       0.3136
Mean shift       

## Good Additional Data

In [14]:
run_partition_tests(good_model, X_additional)


=== Language requirement met ===
Language requirement met: n=7011, mean risk=0.303
Language requirement not met: n=5043, mean risk=0.326
  [FAIL] Language requirement compliance has big impact on predicted risk.

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=3350, mean risk=0.324
Highest 20% language-related score: n=4086, mean risk=0.307
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=2651, mean risk=0.320
At least one attitude/motivation judgement recorded: n=9994, mean risk=0.313
  [OK] 

=== Subjective communication judgement ===
Has communication judgement recorded: n=6643, mean risk=0.312
No communication judgement recorded: n=6002, mean risk=0.317
  [OK] 

=== Subjective appearance/presentation judgements ===
No appearance/presentation judgement recorded: n=5341, mean risk=0.315
At least one appearance/presentation judgement recorded: n=7304, mean risk=0.313
  [OK] 

=== Subjective behavi

In [15]:
run_metamorphic_tests(good_model, X_additional)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.3142
Mean score (flipped)  :       0.3142
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.3142
Mean score (flipped)  :       0.3142
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.3142
Mean score (flipped)  :       0.3142
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Attitude/motivation: baseline -> NONE recorded ===
Mean score (baseline) :       0.3142
Mean score (flipped)  :       0.3142
Mean shift       

## Bad Original Data

In [16]:
run_partition_tests(bad_model, X)


=== Language requirement met ===
Language requirement met: n=71680, mean risk=0.293
Language requirement not met: n=52185, mean risk=0.309
  [OK] 

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=34283, mean risk=0.299
Highest 20% language-related score: n=42352, mean risk=0.303
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=27316, mean risk=0.294
At least one attitude/motivation judgement recorded: n=102684, mean risk=0.303
  [OK] 

=== Subjective communication judgement ===
Has communication judgement recorded: n=68601, mean risk=0.307
No communication judgement recorded: n=61399, mean risk=0.295
  [OK] 

=== Subjective appearance/presentation judgements ===
No appearance/presentation judgement recorded: n=55300, mean risk=0.294
At least one appearance/presentation judgement recorded: n=74700, mean risk=0.306
  [OK] 

=== Subjective behavioural/capacity judments ===
No behavioural/capacity judg

In [17]:
run_metamorphic_tests(bad_model, X)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.3011
Mean score (flipped)  :       0.3011
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.3011
Mean score (flipped)  :       0.3011
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.3011
Mean score (flipped)  :       0.3011
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Attitude/motivation: baseline -> NONE recorded ===
Mean score (baseline) :       0.3011
Mean score (flipped)  :       0.3011
Mean shift       

## Bad Additional Data

In [18]:
run_partition_tests(bad_model, X_additional)


=== Language requirement met ===
Language requirement met: n=7011, mean risk=0.292
Language requirement not met: n=5043, mean risk=0.311
  [OK] 

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=3350, mean risk=0.301
Highest 20% language-related score: n=4086, mean risk=0.303
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=2651, mean risk=0.292
At least one attitude/motivation judgement recorded: n=9994, mean risk=0.304
  [OK] 

=== Subjective communication judgement ===
Has communication judgement recorded: n=6643, mean risk=0.307
No communication judgement recorded: n=6002, mean risk=0.295
  [OK] 

=== Subjective appearance/presentation judgements ===
No appearance/presentation judgement recorded: n=5341, mean risk=0.294
At least one appearance/presentation judgement recorded: n=7304, mean risk=0.307
  [OK] 

=== Subjective behavioural/capacity judments ===
No behavioural/capacity judgement recor

In [19]:
run_metamorphic_tests(bad_model, X_additional)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.3014
Mean score (flipped)  :       0.3014
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.3014
Mean score (flipped)  :       0.3014
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.3014
Mean score (flipped)  :       0.3014
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Attitude/motivation: baseline -> NONE recorded ===
Mean score (baseline) :       0.3014
Mean score (flipped)  :       0.3014
Mean shift       